# Fine-tune WangchanBERTa for 9-Category Thai News Classification (GPU Accelerated)

This notebook downloads **ThaiSum** and **Prachathai-67k**, tokenizes the Thai text, and fine-tunes **WangchanBERTa** (`airesearch/wangchanberta-base-att-spm-uncased`) for 9-category news classification on a free GPU (Google Colab T4 or Kaggle).

### Target 9 Categories:
1. `politics` (การเมือง)
2. `economy` (เศรษฐกิจ)
3. `technology` (เทคโนโลยี)
4. `health` (สุขภาพ)
5. `environment` (สิ่งแวดล้อม)
6. `sports` (กีฬา)
7. `entertainment` (บันเทิง)
8. `society` (สังคม)
9. `world` (ต่างประเทศ)

In [ ]:
# 1. Install Dependencies
!pip install -q transformers torch sentencepiece accelerate pythainlp scikit-learn tqdm httpx protobuf

In [ ]:
# 2. Check GPU
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3. Download ThaiSum & Prachathai datasets
import os, httpx, zipfile, io
from pathlib import Path

os.makedirs("datasets", exist_ok=True)

val_url = "https://nakhun-chumpolsathien.oss-us-west-1.aliyuncs.com/thaisum/validation_set.csv"
test_url = "https://nakhun-chumpolsathien.oss-us-west-1.aliyuncs.com/thaisum/test_set.csv"

for url, name in [(val_url, "datasets/thaisum_val.csv"), (test_url, "datasets/thaisum_test.csv")]:
    if not os.path.exists(name):
        print(f"Downloading {name}...")
        with httpx.stream("GET", url, follow_redirects=True, timeout=120.0) as resp:
            with open(name, "wb") as f:
                for chunk in resp.iter_bytes(65536):
                    f.write(chunk)

print("Datasets ready!")

In [ ]:
# 4. Data Preparation & Ground-Truth Labeling
import csv, random, re
from urllib.parse import unquote

CATEGORIES = [
    "economy", "entertainment", "environment", "health", "politics",
    "society", "sports", "technology", "world"
]
ID2LABEL = {i: cat for i, cat in enumerate(CATEGORIES)}
LABEL2ID = {cat: i for i, cat in enumerate(CATEGORIES)}

def label_article(item_type, item_tags, url, title=""):
    u = unquote(url).lower()
    t = f"{item_type} {item_tags} {title}".lower()
    if "/sport" in u or "ฟุตบอล" in t or "กีฬา" in t:
        return "sports"
    if "/entertain" in u or "บันเทิง" in t or "ดารา" in t or "ละคร" in t:
        return "entertainment"
    if "/health" in u or "สุขภาพ" in t or "โควิด" in t or "วัคซีน" in t or "แพทย์" in t:
        return "health"
    if "/tech" in u or "เทคโนโลยี" in t or "ไอที" in t or "สมาร์ทโฟน" in t:
        return "technology"
    if "/economy" in u or "/business" in u or "เศรษฐกิจ" in t or "หุ้น" in t or "การเงิน" in t:
        return "economy"
    if "/environment" in u or "สิ่งแวดล้อม" in u or "โลกร้อน" in t or "pm2.5" in t:
        return "environment"
    if "/politic" in u or "การเมือง" in t or "นายก" in t or "ครม." in t or "เลือกตั้ง" in t:
        return "politics"
    if "/world" in u or "/foreign" in u or "ต่างประเทศ" in t:
        return "world"
    if "/society" in u or "/lifestyle" in u or "สังคม" in t:
        return "society"
    return None

samples = {cat: [] for cat in CATEGORIES}
for fpath in ["datasets/thaisum_val.csv", "datasets/thaisum_test.csv"]:
    with open(fpath, "r", encoding="utf-8", errors="replace") as f:
        for row in csv.DictReader(f):
            title = (row.get("title") or "").strip()
            cat = label_article(row.get("type", ""), row.get("tags", ""), row.get("url", ""), title=title)
            if cat:
                summary = (row.get("summary") or "").strip()
                body = (row.get("body") or "").strip()
                snippet = summary if summary else body[:300]
                samples[cat].append(f"{title} {snippet}")

X, y = [], []
for cat in CATEGORIES:
    items = samples[cat]
    sampled = random.sample(items, min(1000, len(items)))
    X.extend(sampled)
    y.extend([LABEL2ID[cat]] * len(sampled))

print(f"Total Balanced Samples: {len(X):,}")

In [ ]:
# 5. Fine-Tune WangchanBERTa with HuggingFace Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

model_name = "airesearch/wangchanberta-base-att-spm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9, id2label=ID2LABEL, label2id=LABEL2ID)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=128)

class TorchDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=TorchDataset(train_encodings, y_train),
    eval_dataset=TorchDataset(test_encodings, y_test),
    compute_metrics=compute_metrics,
)

trainer.train()
model.save_pretrained("./wangchanberta_news_classifier")
tokenizer.save_pretrained("./wangchanberta_news_classifier")
print("Model saved to ./wangchanberta_news_classifier")